# Chapter 4: Visualizations and Reports

This notebook generates:
- Performance comparison charts (Figure 4.1-4.4)
- Excel reports with formatted tables
- Publication-quality PNG figures at 300 DPI

In [1]:
import subprocess
import sys

# Install required packages
packages = ["numpy>=1.24", "torch>=2.0", "pyyaml>=6.0", "scipy>=1.10", "matplotlib>=3.7", "openpyxl>=3.1"]
for package in packages:
    try:
        package_name = package.split(">=")[0]
        __import__(package_name)
    except ImportError:
        print(f"Installing {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

print("✅ All dependencies installed")

Installing pyyaml>=6.0...
✅ All dependencies installed


## Setup and Imports

In [2]:
import os
import csv
import numpy as np
from typing import List, Dict, Tuple
from pathlib import Path

# Visualization imports
try:
    import matplotlib.pyplot as plt
    import matplotlib
    matplotlib.rcParams['figure.dpi'] = 300  # type: ignore[index]
    MATPLOTLIB_AVAILABLE = True
except ImportError:
    MATPLOTLIB_AVAILABLE = False
    print("⚠️  matplotlib not available. Charts will not be generated.")

# Excel export
try:
    from openpyxl import Workbook
    from openpyxl.styles import Font, Alignment, PatternFill
    OPENPYXL_AVAILABLE = True
except ImportError:
    OPENPYXL_AVAILABLE = False
    print("⚠️  openpyxl not available. Excel reports will not be generated.")

print(f"✅ Imports loaded (matplotlib: {MATPLOTLIB_AVAILABLE}, openpyxl: {OPENPYXL_AVAILABLE})")

✅ Imports loaded (matplotlib: True, openpyxl: True)


## Data Loading Utilities

In [3]:
def load_csv_results(filepath: str) -> Dict[str, List[float]]:  # type: ignore[no-untyped-def]
    """Load evaluation results from CSV file."""
    data: Dict[str, List] = {  # type: ignore[assignment]
        'episode': [],
        'avg_queue': [],
        'max_queue': [],
        'min_queue': [],
        'avg_wait': [],
        'max_wait': [],
        'avg_speed': [],
        'cumulative_reward': [],
        'vehicles_passed': []
    }
    
    if not os.path.exists(filepath):
        print(f"⚠️  File not found: {filepath}")
        return data  # type: ignore[return-value]
    
    with open(filepath, 'r') as f:
        reader = csv.DictReader(f)
        for row in reader:
            data['episode'].append(int(row['Episode']))  # type: ignore[index]
            data['avg_queue'].append(float(row['Avg_Queue']))  # type: ignore[index]
            data['max_queue'].append(float(row['Max_Queue']))  # type: ignore[index]
            data['min_queue'].append(float(row['Min_Queue']))  # type: ignore[index]
            data['avg_wait'].append(float(row['Avg_Wait_Time']))  # type: ignore[index]
            data['max_wait'].append(float(row['Max_Wait_Time']))  # type: ignore[index]
            data['avg_speed'].append(float(row['Avg_Speed']))  # type: ignore[index]
            data['cumulative_reward'].append(float(row['Cumulative_Reward']))  # type: ignore[index]
            data['vehicles_passed'].append(int(row['Vehicles_Passed']))  # type: ignore[index]
    
    print(f"✅ Loaded {len(data['episode'])} episodes from {filepath}")
    return data

print("✅ Data loading utilities defined")

✅ Data loading utilities defined


## Visualization Functions

In [4]:
def create_improvement_metrics_plot(  # type: ignore[no-untyped-def]
    dqn_data: Dict[str, List[float]],
    fixed_data: Dict[str, List[float]],
    output_path: str = "outputs/chapter4/improvement_metrics.png"
) -> None:  # type: ignore[no-untyped-def]
    """Figure 4.4: Improvement percentage chart."""
    if not MATPLOTLIB_AVAILABLE:
        print("⚠️  Skipping chart generation (matplotlib not available)")
        return
    
    metrics = ['avg_queue', 'avg_wait', 'max_queue']
    labels = ['Avg Queue\\nReduction', 'Avg Wait Time\\nReduction', 'Max Queue\\nReduction']
    
    improvements: list[float] = []  # type: ignore[assignment]
    for metric in metrics:
        dqn_mean = float(np.mean(dqn_data[metric]))  # type: ignore[index]
        fixed_mean = float(np.mean(fixed_data[metric]))  # type: ignore[index]
        improvement = ((fixed_mean - dqn_mean) / fixed_mean) * 100
        improvements.append(improvement)
    
    # Add speed improvement (opposite direction)
    speed_improvement = float(((np.mean(dqn_data['avg_speed']) - np.mean(fixed_data['avg_speed'])) /  # type: ignore[index]
                        np.mean(fixed_data['avg_speed'])) * 100)  # type: ignore[index]
    improvements.append(speed_improvement)
    labels.append('Avg Speed\\nIncrease')
    
    colors = ['#2ecc71' if imp > 0 else '#e74c3c' for imp in improvements]
    
    fig, ax = plt.subplots(figsize=(10, 6))  # type: ignore[call-overload]
    
    bars = ax.bar(range(len(labels)), improvements, color=colors, alpha=0.8, edgecolor='black')  # type: ignore[call-overload]
    
    ax.set_xlabel('Metric', fontsize=12, fontweight='bold')  # type: ignore[call-overload]
    ax.set_ylabel('Improvement (%)', fontsize=12, fontweight='bold')  # type: ignore[call-overload]
    ax.set_title('Figure 4.4: DQN Performance Improvement Over Fixed-Time',
                 fontsize=14, fontweight='bold', pad=20)  # type: ignore[call-overload]
    ax.set_xticks(range(len(labels)))  # type: ignore[call-overload]
    ax.set_xticklabels(labels)  # type: ignore[call-overload]
    ax.axhline(y=0, color='black', linestyle='-', linewidth=0.8)  # type: ignore[call-overload]
    ax.grid(axis='y', alpha=0.3, linestyle='--')  # type: ignore[call-overload]
    
    # Add value labels on bars
    for i, (bar, val) in enumerate(zip(bars, improvements)):
        height = bar.get_height()  # type: ignore[call-overload]
        ax.text(bar.get_x() + bar.get_width()/2., height,  # type: ignore[call-overload]
                f'{val:.1f}%', ha='center', va='bottom' if val > 0 else 'top',  # type: ignore[call-overload]
                fontweight='bold', fontsize=10)  # type: ignore[call-overload]
    
    plt.tight_layout()  # type: ignore[no-untyped-call]
    Path(output_path).parent.mkdir(parents=True, exist_ok=True)  # type: ignore[call-overload]
    plt.savefig(output_path, dpi=300, bbox_inches='tight')  # type: ignore[no-untyped-call]
    plt.close()  # type: ignore[no-untyped-call]
    
    print(f"✅ Figure 4.4 saved to: {output_path}")

print("✅ Visualization functions defined")

✅ Visualization functions defined


In [5]:
def create_comparison_bar_chart(  # type: ignore[no-untyped-def]
    dqn_data: Dict[str, List[float]],
    fixed_data: Dict[str, List[float]],
    output_path: str = "outputs/chapter4/comparison_bar_chart.png"
) -> None:  # type: ignore[no-untyped-def]
    """Figure 4.1: Comparison bar chart for key metrics."""
    if not MATPLOTLIB_AVAILABLE:
        print("⚠️  Skipping chart generation (matplotlib not available)")
        return
    
    metrics = {
        'avg_queue': ('Avg Queue Length', 'vehicles'),
        'avg_wait': ('Avg Waiting Time', 'seconds'),
        'max_queue': ('Max Queue Length', 'vehicles')
    }
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))  # type: ignore[call-overload]
    
    for idx, (key, (title, unit)) in enumerate(metrics.items()):
        dqn_values = dqn_data[key]  # type: ignore[index]
        fixed_values = fixed_data[key]  # type: ignore[index]
        
        dqn_mean = float(np.mean(dqn_values))
        fixed_mean = float(np.mean(fixed_values))
        dqn_std = float(np.std(dqn_values))
        fixed_std = float(np.std(fixed_values))
        
        x_pos = np.arange(2)
        bars = axes[idx].bar(x_pos, [dqn_mean, fixed_mean], yerr=[dqn_std, fixed_std],  # type: ignore[call-overload]
                            color=['#3498db', '#e74c3c'], alpha=0.8, capsize=5, edgecolor='black')
        
        axes[idx].set_ylabel(f'{title} ({unit})', fontsize=10, fontweight='bold')  # type: ignore[call-overload]
        axes[idx].set_title(title, fontsize=11, fontweight='bold')  # type: ignore[call-overload]
        axes[idx].set_xticks(x_pos)  # type: ignore[call-overload]
        axes[idx].set_xticklabels(['DQN', 'Fixed-Time'])  # type: ignore[call-overload]
        axes[idx].grid(axis='y', alpha=0.3, linestyle='--')  # type: ignore[call-overload]
        
        # Add value labels
        for i, (bar, val) in enumerate(zip(bars, [dqn_mean, fixed_mean])):
            axes[idx].text(bar.get_x() + bar.get_width()/2., val,  # type: ignore[call-overload]
                          f'{val:.1f}', ha='center', va='bottom', fontweight='bold')  # type: ignore[call-overload]
    
    fig.suptitle('Figure 4.1: Performance Metrics Comparison', fontsize=14, fontweight='bold', y=1.02)  # type: ignore[call-overload]
    plt.tight_layout()  # type: ignore[no-untyped-call]
    Path(output_path).parent.mkdir(parents=True, exist_ok=True)  # type: ignore[call-overload]
    plt.savefig(output_path, dpi=300, bbox_inches='tight')  # type: ignore[no-untyped-call]
    plt.close()  # type: ignore[no-untyped-call]
    
    print(f"✅ Figure 4.1 saved to: {output_path}")

def create_performance_tracking_plot(  # type: ignore[no-untyped-def]
    dqn_data: Dict[str, List[float]],
    fixed_data: Dict[str, List[float]],
    output_path: str = "outputs/chapter4/performance_tracking.png"
) -> None:  # type: ignore[no-untyped-def]
    """Figure 4.2: Performance tracking over episodes."""
    if not MATPLOTLIB_AVAILABLE:
        print("⚠️  Skipping chart generation (matplotlib not available)")
        return
    
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))  # type: ignore[call-overload]
    
    # Queue length tracking
    axes[0, 0].plot(dqn_data['episode'], dqn_data['avg_queue'], 'o-', color='#3498db', label='DQN', linewidth=2)  # type: ignore[call-overload]
    axes[0, 0].plot(fixed_data['episode'], fixed_data['avg_queue'], 's-', color='#e74c3c', label='Fixed-Time', linewidth=2)  # type: ignore[call-overload]
    axes[0, 0].set_ylabel('Avg Queue Length (vehicles)', fontsize=10, fontweight='bold')  # type: ignore[call-overload]
    axes[0, 0].set_title('Average Queue Length Over Episodes', fontsize=11, fontweight='bold')  # type: ignore[call-overload]
    axes[0, 0].legend()  # type: ignore[call-overload]
    axes[0, 0].grid(alpha=0.3, linestyle='--')  # type: ignore[call-overload]
    
    # Waiting time tracking
    axes[0, 1].plot(dqn_data['episode'], dqn_data['avg_wait'], 'o-', color='#3498db', label='DQN', linewidth=2)  # type: ignore[call-overload]
    axes[0, 1].plot(fixed_data['episode'], fixed_data['avg_wait'], 's-', color='#e74c3c', label='Fixed-Time', linewidth=2)  # type: ignore[call-overload]
    axes[0, 1].set_ylabel('Avg Waiting Time (seconds)', fontsize=10, fontweight='bold')  # type: ignore[call-overload]
    axes[0, 1].set_title('Average Waiting Time Over Episodes', fontsize=11, fontweight='bold')  # type: ignore[call-overload]
    axes[0, 1].legend()  # type: ignore[call-overload]
    axes[0, 1].grid(alpha=0.3, linestyle='--')  # type: ignore[call-overload]
    
    # Speed tracking
    axes[1, 0].plot(dqn_data['episode'], dqn_data['avg_speed'], 'o-', color='#3498db', label='DQN', linewidth=2)  # type: ignore[call-overload]
    axes[1, 0].plot(fixed_data['episode'], fixed_data['avg_speed'], 's-', color='#e74c3c', label='Fixed-Time', linewidth=2)  # type: ignore[call-overload]
    axes[1, 0].set_ylabel('Avg Speed (m/s)', fontsize=10, fontweight='bold')  # type: ignore[call-overload]
    axes[1, 0].set_title('Average Speed Over Episodes', fontsize=11, fontweight='bold')  # type: ignore[call-overload]
    axes[1, 0].legend()  # type: ignore[call-overload]
    axes[1, 0].grid(alpha=0.3, linestyle='--')  # type: ignore[call-overload]
    
    # Reward tracking
    axes[1, 1].plot(dqn_data['episode'], dqn_data['cumulative_reward'], 'o-', color='#3498db', label='DQN', linewidth=2)  # type: ignore[call-overload]
    axes[1, 1].plot(fixed_data['episode'], fixed_data['cumulative_reward'], 's-', color='#e74c3c', label='Fixed-Time', linewidth=2)  # type: ignore[call-overload]
    axes[1, 1].set_ylabel('Cumulative Reward', fontsize=10, fontweight='bold')  # type: ignore[call-overload]
    axes[1, 1].set_title('Cumulative Reward Over Episodes', fontsize=11, fontweight='bold')  # type: ignore[call-overload]
    axes[1, 1].legend()  # type: ignore[call-overload]
    axes[1, 1].grid(alpha=0.3, linestyle='--')  # type: ignore[call-overload]
    
    # Set x-label for bottom plots
    for ax in axes[1, :]:
        ax.set_xlabel('Episode', fontsize=10, fontweight='bold')  # type: ignore[call-overload]
    
    fig.suptitle('Figure 4.2: Performance Tracking Over Episodes', fontsize=14, fontweight='bold', y=0.995)  # type: ignore[call-overload]
    plt.tight_layout()  # type: ignore[no-untyped-call]
    Path(output_path).parent.mkdir(parents=True, exist_ok=True)  # type: ignore[call-overload]
    plt.savefig(output_path, dpi=300, bbox_inches='tight')  # type: ignore[no-untyped-call]
    plt.close()  # type: ignore[no-untyped-call]
    
    print(f"✅ Figure 4.2 saved to: {output_path}")

def create_queue_distribution_plot(  # type: ignore[no-untyped-def]
    dqn_data: Dict[str, List[float]],
    fixed_data: Dict[str, List[float]],
    output_path: str = "outputs/chapter4/queue_distribution.png"
) -> None:  # type: ignore[no-untyped-def]
    """Figure 4.3: Queue length distribution comparison."""
    if not MATPLOTLIB_AVAILABLE:
        print("⚠️  Skipping chart generation (matplotlib not available)")
        return
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))  # type: ignore[call-overload]
    
    # DQN histogram
    axes[0].hist(dqn_data['avg_queue'], bins=10, color='#3498db', alpha=0.7, edgecolor='black')  # type: ignore[call-overload]
    axes[0].set_xlabel('Queue Length (vehicles)', fontsize=10, fontweight='bold')  # type: ignore[call-overload]
    axes[0].set_ylabel('Frequency', fontsize=10, fontweight='bold')  # type: ignore[call-overload]
    axes[0].set_title('DQN Queue Length Distribution', fontsize=11, fontweight='bold')  # type: ignore[call-overload]
    axes[0].grid(axis='y', alpha=0.3, linestyle='--')  # type: ignore[call-overload]
    
    # Fixed-Time histogram
    axes[1].hist(fixed_data['avg_queue'], bins=10, color='#e74c3c', alpha=0.7, edgecolor='black')  # type: ignore[call-overload]
    axes[1].set_xlabel('Queue Length (vehicles)', fontsize=10, fontweight='bold')  # type: ignore[call-overload]
    axes[1].set_ylabel('Frequency', fontsize=10, fontweight='bold')  # type: ignore[call-overload]
    axes[1].set_title('Fixed-Time Queue Length Distribution', fontsize=11, fontweight='bold')  # type: ignore[call-overload]
    axes[1].grid(axis='y', alpha=0.3, linestyle='--')  # type: ignore[call-overload]
    
    fig.suptitle('Figure 4.3: Queue Length Distribution', fontsize=14, fontweight='bold', y=1.02)  # type: ignore[call-overload]
    plt.tight_layout()  # type: ignore[no-untyped-call]
    Path(output_path).parent.mkdir(parents=True, exist_ok=True)  # type: ignore[call-overload]
    plt.savefig(output_path, dpi=300, bbox_inches='tight')  # type: ignore[no-untyped-call]
    plt.close()  # type: ignore[no-untyped-call]
    
    print(f"✅ Figure 4.3 saved to: {output_path}")

print("✅ All visualization functions defined")

✅ All visualization functions defined


## Excel Report Generation

In [6]:
def generate_excel_report(
    dqn_data: Dict[str, List[float]],
    fixed_data: Dict[str, List[float]],
    output_path: str = "outputs/chapter4/Chapter4_Results_Report.xlsx"
):
    """Generate comprehensive Excel report with all tables."""
    if not OPENPYXL_AVAILABLE:
        print("⚠️  Skipping Excel generation (openpyxl not available)")
        return
    
    wb = Workbook()
    wb.remove(wb.active)  # Remove default sheet
    
    # Sheet 1: Summary Statistics
    ws1 = wb.create_sheet("Summary Statistics")
    headers = ['Metric', 'DQN Mean', 'DQN Std', 'Fixed Mean', 'Fixed Std', 'Improvement %']
    ws1.append(headers)
    
    metrics = [
        ('Average Queue Length', 'avg_queue', False),
        ('Max Queue Length', 'max_queue', False),
        ('Average Waiting Time', 'avg_wait', False),
        ('Average Speed', 'avg_speed', True),
        ('Cumulative Reward', 'cumulative_reward', True)
    ]
    
    for label, key, higher_better in metrics:
        dqn_mean = np.mean(dqn_data[key])
        dqn_std = np.std(dqn_data[key])
        fixed_mean = np.mean(fixed_data[key])
        fixed_std = np.std(fixed_data[key])
        
        if higher_better:
            improvement = ((dqn_mean - fixed_mean) / fixed_mean) * 100
        else:
            improvement = ((fixed_mean - dqn_mean) / fixed_mean) * 100
        
        ws1.append([label, f"{dqn_mean:.2f}", f"{dqn_std:.2f}", 
                   f"{fixed_mean:.2f}", f"{fixed_std:.2f}", f"{improvement:.1f}%"])
    
    # Sheet 2: Raw DQN Data
    ws2 = wb.create_sheet("DQN Results")
    ws2.append(['Episode', 'Avg Queue', 'Max Queue', 'Avg Wait', 'Avg Speed', 'Cumulative Reward'])
    
    for i in range(len(dqn_data['episode'])):
        ws2.append([
            dqn_data['episode'][i],
            f"{dqn_data['avg_queue'][i]:.2f}",
            f"{dqn_data['max_queue'][i]:.2f}",
            f"{dqn_data['avg_wait'][i]:.2f}",
            f"{dqn_data['avg_speed'][i]:.2f}",
            f"{dqn_data['cumulative_reward'][i]:.2f}"
        ])
    
    # Sheet 3: Raw Fixed-Time Data
    ws3 = wb.create_sheet("Fixed-Time Results")
    ws3.append(['Episode', 'Avg Queue', 'Max Queue', 'Avg Wait', 'Avg Speed', 'Cumulative Reward'])
    
    for i in range(len(fixed_data['episode'])):
        ws3.append([
            fixed_data['episode'][i],
            f"{fixed_data['avg_queue'][i]:.2f}",
            f"{fixed_data['max_queue'][i]:.2f}",
            f"{fixed_data['avg_wait'][i]:.2f}",
            f"{fixed_data['avg_speed'][i]:.2f}",
            f"{fixed_data['cumulative_reward'][i]:.2f}"
        ])
    
    Path(output_path).parent.mkdir(parents=True, exist_ok=True)
    wb.save(output_path)
    
    print(f"✅ Excel report saved to: {output_path}")

print("✅ Excel report function defined")

✅ Excel report function defined


## Generate All Visualizations and Reports

Run this cell to create all figures and reports from evaluation results.

In [7]:
# Configuration
DQN_RESULTS_CSV = "outputs/chapter4/dqn_results.csv"
FIXED_RESULTS_CSV = "outputs/chapter4/fixed_results.csv"
OUTPUT_DIR = "outputs/chapter4"

print("\n" + "="*60)
print("GENERATING CHAPTER 4 VISUALIZATIONS AND REPORTS")
print("="*60 + "\n")

# Load results
dqn_data = load_csv_results(DQN_RESULTS_CSV)
fixed_data = load_csv_results(FIXED_RESULTS_CSV)

if len(dqn_data['episode']) == 0 or len(fixed_data['episode']) == 0:
    print("⚠️  No data found. Please run chapter4_evaluation.ipynb first.")
else:
    print(f"\n📊 Generating visualizations...\n")
    
    # Generate all figures
    create_comparison_bar_chart(dqn_data, fixed_data)  # type: ignore[reportCallIssue]
    create_performance_tracking_plot(dqn_data, fixed_data)  # type: ignore[reportCallIssue]
    create_queue_distribution_plot(dqn_data, fixed_data)  # type: ignore[reportCallIssue]
    create_improvement_metrics_plot(dqn_data, fixed_data)  # type: ignore[reportCallIssue]
    
    # Generate Excel report
    print(f"\n📑 Generating Excel report...\n")
    generate_excel_report(dqn_data, fixed_data)
    
    print("\n" + "="*60)
    print("✅ ALL VISUALIZATIONS AND REPORTS GENERATED")
    print("="*60)
    print(f"\nOutput location: {OUTPUT_DIR}/")
    print("\nGenerated files:")
    print("  📊 comparison_bar_chart.png (Figure 4.1)")
    print("  📊 performance_tracking.png (Figure 4.2)")
    print("  📊 queue_distribution.png (Figure 4.3)")
    print("  📊 improvement_metrics.png (Figure 4.4)")
    print("  📑 Chapter4_Results_Report.xlsx")
    print("\n✨ Ready for thesis writing!")


GENERATING CHAPTER 4 VISUALIZATIONS AND REPORTS

✅ Loaded 2 episodes from outputs/chapter4/dqn_results.csv
✅ Loaded 2 episodes from outputs/chapter4/fixed_results.csv

📊 Generating visualizations...

✅ Figure 4.1 saved to: outputs/chapter4/comparison_bar_chart.png
✅ Figure 4.2 saved to: outputs/chapter4/performance_tracking.png
✅ Figure 4.3 saved to: outputs/chapter4/queue_distribution.png
✅ Figure 4.4 saved to: outputs/chapter4/improvement_metrics.png

📑 Generating Excel report...

✅ Excel report saved to: outputs/chapter4/Chapter4_Results_Report.xlsx

✅ ALL VISUALIZATIONS AND REPORTS GENERATED

Output location: outputs/chapter4/

Generated files:
  📊 comparison_bar_chart.png (Figure 4.1)
  📊 performance_tracking.png (Figure 4.2)
  📊 queue_distribution.png (Figure 4.3)
  📊 improvement_metrics.png (Figure 4.4)
  📑 Chapter4_Results_Report.xlsx

✨ Ready for thesis writing!
